# Q1. Object detection and depth estimation                             20+5+20 = 45 points
 Use L515 camera in Lab to acquire rgb video and depth video (e.g. 30 s) of swinging load in lab (rgb_train_val_swingingload.avi; depth_train_val_swingingload.avi). Extract each frames from rgb video. Use the extracted images to train and validate YOLO model (save as YOLO_swinging_load.pt). Record test_swingingload.avi (e.g. 5 s). Extract frames and use them as test data set. Test your YOLO model (YOLO_swinging_load.pt) on this test data set.

(a) Show the performance of your model on test data set by plotting relevant KPIs.

(b) Sample five random images from test data video (test_swingingload.avi). Find the center, and coordinates of bounding box. Display them on the each test image.

(c) realsense_acq.ipynb contains code that streams, rgb, and depth video. Based on this code, extract the frames from depth_train_val_swingingload.avi. Each pixel in each frame consists of depth information of corresponding rgb images. As described in Q1.b find the center of bounding box of ALL frames in  test video (test_swingingload.avi). Display its center and corresponding depth.


References: Use Ultralytics framework for help - https://docs.ultralytics.com/tasks/detect
Realsense - https://dev.realsenseai.com/sdk-2-0-code-samples-wrappers-and-languages/opencv/ 

Data acquisition: realsense_acq.ipynb contains the code to acquire RGB and Depth data from L515.
 
Cell 1: Streams RGB and Depth data

Cell 2: Streams RGB and Depth data, and saves the following:
- session_date_time e.g. session_20260914_1354
  - /color_video. Contains colour RGB video
  - /depth_video. Contains depth video video
  - /depth_raw. Contains raw depth value
  - /depth_scale_json. Contains scaling factor. multiply this scaling factor with raw scale to obtain true depth in meter
  -/rgb_frames. Contains RGB frames that you can use for labeling and object detection
  

In [ ]:
# Q1 Prep:
# Pull in ultralytics pretrained model, train on custom dataset, save model

# Dataset preparation required:
# - Moving all rgb_frames into separate folders in q1/dataset/ 
# - Split training images 90% to keep and move 10% into val/ sequentially to avoid data leakage
# - Label all .png using Labelme to generate Labelme .JSON labels (Took me 6 hours!)
# - Move all labels into labels/ directory
# - Convert all JSON labels to YOLO formatted .txt using conversion script from Gemini
# 
# dataset/
#    |- images/
#        |- train/ -> rgb_0000.png - rgb_0809.png
#        |- test/ -> rgb_0000.png - rgb_0299.png
#        |- val/ -> rgb_0810.png - rgb_0899.png
#    |- labels/
#        |- train/ ->  rgb_0000.txt - rgb_0809.txt
#        |- test/ -> rgb_0000.txt - rgb_0299.txt
#        |- val/ -> rgb_0810.txt - rgb_0899.txt
#    |- dataset.yaml
#    |- labelme2yolo.py

from ultralytics import YOLO

# Want to test yolo26n because this is a model we will be using in SeaBotics, attempting transfer of knowledge from curriculum to real-life
# Higher "tier" model such as yolo26x would have higher accuracy, but takes longer to train and is not small enough for edge computing-
# as desired in SeaBotics project (Luxonis OAK-D Pro AI Camera).

model = YOLO("yolo26n.pt")

results = model.train(
    data="q1/dataset/dataset.yaml",
    epochs=100,
    imgsz=(640, 480), # -- Images are 640x480, not 640x640
    ) 

# Have edited out test/ from dataset.yaml for training to avoid accidentally exposing it to model during training
# Now run model and it will automatically create a best.pt which can be copied and renamed to YOLO_swinging_load.pt in q1/model

In [ ]:
# Q1 A
# Uncomment test directory from dataset.yaml before running

from ultralytics import YOLO

model = YOLO("q1/model/YOLO_swinging_load.pt")

metrics = model.val(data="q1/dataset/dataset.yaml", split="test")

# Plot metrics
print(metrics.box.results_dict) 

In [ ]:
# Q1 B 

from ultralytics import YOLO
import os
import numpy as np

images = []
output_images = []
np.random.seed(69) # Nice
dir1 = "./q1/dataset/images/test"
dir2 = "./q1/inferred"

os.makedirs(dir2, exist_ok=True)

for i in range(5):
    # Generate a random number between 0 and 299
    # Fetch an image at random number position from given image test directory
    # Add that image to images array 

    img_number = np.random.randint(0, 300) # -- amount of image files in test data set, 000 - 299
    image_name = f"rgb_{img_number:04d}.png"
    images.append(os.path.join(dir1, image_name))

# Load YOLO_swinging_load.pt 
model = YOLO("q1/model/YOLO_swinging_load.pt")
counter = 0;

for image in images:
    # Run actual inference using model on each image
    results = model(image)

    # Store inferred image and bounding box in output_images and on new .png files
    output_images.append(results[0])
    results[0].save(os.path.join(dir2,f"{counter}.png"))
    counter += 1
    
for output in output_images:
    # Display inferred images with bounding boxes
    output.show()

# Q3. Multi object segmentation, tracking and counting:                         10  points

Explore different data-set available freely on internet (https://docs.ultralytics.com/datasets ). Get familiar with them. Then choose a data set and corresponding class of your choice (e.g. CIFAR-10, dataset/COCO data set or KITTI data set / person). Then make a video such that multiple instances of objects are in the scene (e.g. 5 people are the scene). Perform 
(a) instance segmentation, 
(b) counting  - total number of object in scene
(c) and tracking of individual instance of object
(d) draw a region - and count the object in that region